# Listening Test — Dataset Samples (8/24)

This notebook prepares the **dataset third** of a listening test. Eight clips come from the ANIMA 53-EDO corpus (this notebook); eight more come from Model A, and eight from Model B (notebooks 10 and 11, same render settings).

Each clip is a **17-bar chord-progression excerpt** rendered from an MPE MIDI file via `src/play_mpe.py` (FluidSynth + FluidR3_GM SoundFont, Rhodes patch). The renderer honours MPE per-channel pitch-bend so every 53-EDO step lands at its exact frequency.

**Design (24 clips total)**:
- **4 styles** × **2 samples each** = 8 per source — `jazz`, `rock`, `blues`, `bossa`.
- **8 transformation types** (one per sample, covering the microtonal vocabulary):
  `type_0_major`, `type_1_neutral`, `type_2_subminor`, `type_3_major`,
  `type_4_minor`, `type_5_major_v2`, `type_5_minor`, `type_6_neutral_n`.
- **3 sources**: `dataset` (this notebook), `model_A`, `model_B` — each under its own folder.

**Survey questions** (per clip):
- **Harmony** — How coherent do you find the harmonic motion?
- **Plausibility** — How plausible is this chord progression for a potential song?
- **Dissonance** — How dissonant is this chord progression?
- **Novelty** — How surprising or novel do you find this progression?

**Output layout**:
- `dataset/audio/listening_test/dataset/*.wav`
- `dataset/listening_test/dataset/midi/*.mid`
- `dataset/listening_test/dataset/manifest.json`

The matching `model_A/` and `model_B/` subfolders are populated by notebooks 10 and 11.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
from IPython.display import Audio, display

# Make the project source importable regardless of launch directory.
_SRC = Path("/home/david/Projects/ANIMA_Microtonal_GPT/src")
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from tokenizer import parse_mpe_midi, MPETokenizer
from play_mpe import render_mpe_to_audio_data

ROOT = _SRC.parent
MIDI_ROOT_BASE = ROOT / "dataset" / "midi_files" / "53_tet_mpe"

# Source tag — used as both the audio/midi subfolder name and the manifest key.
# Sister notebooks (10, 11) mirror this layout with SOURCE = "model_A" / "model_B".
SOURCE = "dataset"

# Audio goes to dataset/audio/listening_test/<SOURCE>/; trimmed MIDI + manifest
# live under dataset/listening_test/<SOURCE>/ so each source is fully isolated.
OUT_AUDIO = ROOT / "dataset" / "audio" / "listening_test" / SOURCE
OUT_DIR   = ROOT / "dataset" / "listening_test" / SOURCE
OUT_MIDI  = OUT_DIR / "midi"
OUT_AUDIO.mkdir(parents=True, exist_ok=True)
OUT_MIDI.mkdir(parents=True, exist_ok=True)

print("MIDI_ROOT_BASE :", MIDI_ROOT_BASE)
print("OUT_AUDIO      :", OUT_AUDIO)
print("OUT_MIDI       :", OUT_MIDI)
assert MIDI_ROOT_BASE.exists(), f"Not found: {MIDI_ROOT_BASE}"


## Curated 8-song selection — 4 styles × 2 samples, 8 transformations

Eight songs chosen from the iReal corpus — **2 per style** across `jazz`, `blues`, `bossa`, and `rock` — and each paired with a **distinct** 53-EDO transformation so the set probes all eight tuning regimes in the listening test.

| # | Song | Key | Style | Transformation |
|---|------|-----|-------|----------------|
| 1 | Autumn Leaves            | F  | jazz  | `type_0_major` (12-TET reference) |
| 2 | Misty                    | F  | jazz  | `type_5_minor` |
| 3 | Crossroads (Cross Road Blues) | F  | blues | `type_2_subminor` |
| 4 | Bessie's Blues           | F  | blues | `type_6_neutral_n` |
| 5 | Wave                     | F  | bossa | `type_1_neutral` |
| 6 | Só Tinha De Ser Com Você | F  | bossa | `type_3_major` |
| 7 | Something (Beatles)      | F  | rock  | `type_5_major_v2` |
| 8 | Fix You (Coldplay)       | F  | rock  | `type_4_minor` |

All songs are resolved against the `_F_` key variant to keep timbral/register context comparable across clips.


In [ ]:
# Eight entries: 2 per style across {jazz, blues, bossa, rock}, each on a
# distinct transformation from the 8-type set. Style tags match the STYLE_*
# tokens Model A / Model B are conditioned on in notebooks 10 / 11.
SELECTION = [
    # (display_name, filename_glob_fragment, style_tag, transformation_type)
    ("Autumn Leaves",              "_Autumn Leaves_F_",                     "jazz",  "type_0_major"),
    ("Misty",                      "_Misty_F_",                             "jazz",  "type_5_minor"),
    ("Crossroads",                 "_Crossroads aka Cross Road Blues_F_",   "blues", "type_2_subminor"),
    ("Bessies Blues",              "_Bessies Blues_F_",                     "blues", "type_6_neutral_n"),
    ("Wave",                       "_Wave_F_",                              "bossa", "type_1_neutral"),
    ("So Tinha De Ser Com Voce",   "_So Tinha De Ser Com Vo\u00e7e_F_",     "bossa", "type_3_major"),
    ("Something",                  "_Something_F_",                         "rock",  "type_5_major_v2"),
    ("Fix You",                    "_Fix You_F_",                           "rock",  "type_4_minor"),
]

assert len(SELECTION) == 8
assert {s[2] for s in SELECTION} == {"jazz", "blues", "bossa", "rock"}
assert len({s[3] for s in SELECTION}) == 8  # every sample hits a different transformation


def resolve(glob_fragment: str, transformation: str) -> Path:
    """Find the MIDI whose stem contains `glob_fragment` inside the given transformation folder."""
    midi_dir = MIDI_ROOT_BASE / transformation
    hits = sorted(
        p for p in midi_dir.iterdir()
        if p.suffix == ".mid" and glob_fragment in p.name
    )
    if not hits:
        raise FileNotFoundError(f"No MIDI matching {glob_fragment!r} in {midi_dir}")
    # Prefer the shortest matching name (no " 1", " 2" variants).
    hits.sort(key=lambda p: (len(p.name), p.name))
    return hits[0]

resolved = []
for name, frag, style, ttype in SELECTION:
    path = resolve(frag, ttype)
    resolved.append((name, path, style, ttype))
    print(f"{name:26s} [{style:5s}] [{ttype:17s}] -> {path.name}")


## Inspect chord structure

Each MIDI is parsed into a list of chord events — `{onset_beats, duration_beats, notes:[{step_53, velocity}]}` — using the read-only `parse_mpe_midi` from `tokenizer.py`. This lets us trim cleanly on chord boundaries.

In [ ]:
# Peek at one song so we understand the chord-event format.
name, path, _, ttype = resolved[0]
chords = parse_mpe_midi(path)
total_beats = chords[-1]["onset_beats"] + chords[-1]["duration_beats"]
print(f"{name} [{ttype}]: {len(chords)} chords, total={total_beats:.1f} beats")
for c in chords[:4]:
    steps = [n["step_53"] for n in c["notes"]]
    print(f"  onset={c['onset_beats']:6.2f}  dur={c['duration_beats']:4.1f}  steps={steps}")

## Trim to 16 bars & render at 160 BPM

iReal-derived MIDI uses 4/4 throughout and 1 beat = 1 quarter note → 16 bars = **64 beats**. We trim every chord whose onset is strictly before `max_beats` and clip the final chord's duration so it never rings past the window.

We write the trimmed MIDI at **160 BPM** (instead of the 120 BPM the clips were packed at) so the progressions have a motion-forward feel rather than dragging.

In [ ]:
N_BARS = 17
BEATS_PER_BAR = 4
MAX_BEATS = N_BARS * BEATS_PER_BAR

TEMPO_BPM = 180                 # render at 160 BPM — jazz/pop standards sound alive here
RENDER_SPEED = 1.0              # no further tempo scaling at render time
RENDER_WAVEFORM = "rhodes"      # sine, triangle, sawtooth, square, piano, rhodes 
RENDER_REVERB = 33              # light hall ambience (0 = dry)
SAMPLE_RATE = 44100

    
def trim_chords_to_bars(chords, max_beats=MAX_BEATS):
    """Keep chords starting before `max_beats`; clip the final chord's ring-out."""
    kept = []
    for c in chords:
        if c["onset_beats"] >= max_beats:
            break
        end = c["onset_beats"] + c["duration_beats"]
        if end > max_beats:
            c = dict(c, duration_beats=round(max_beats - c["onset_beats"], 4))
        kept.append(c)
    return kept


def write_trimmed_midi(chords, out_path: Path, tempo_bpm=TEMPO_BPM):
    """Write the chord list back to MPE MIDI at the requested tempo."""
    tok = MPETokenizer()
    tok.chords_to_midi(chords, out_path, tpb=960, tempo_bpm=tempo_bpm)


def render_wav(midi_path: Path, wav_path: Path):
    """Render an MPE MIDI file to a WAV at SAMPLE_RATE using sine synthesis.

    Delegates to play_mpe.render_mpe_to_audio_data's own `save_path` so the
    (channels, samples) → (samples, channels) transpose is handled correctly.
    """
    audio, sr = render_mpe_to_audio_data(
        str(midi_path),
        sample_rate=SAMPLE_RATE,
        speed=RENDER_SPEED,
        waveform=RENDER_WAVEFORM,
        reverb=RENDER_REVERB,
        save_path=str(wav_path),
    )
    if audio is None:
        raise RuntimeError(f"render_mpe_to_audio_data returned None for {midi_path}")
    n_samples = audio.shape[-1] if audio.ndim == 2 else len(audio)
    return sr, n_samples

In [ ]:
manifest_entries = []

for idx, (display_name, src_midi, style, ttype) in enumerate(resolved, start=1):
    # 1) parse → trim to 17 bars
    chords = parse_mpe_midi(src_midi)
    trimmed = trim_chords_to_bars(chords)
    actual_end = trimmed[-1]["onset_beats"] + trimmed[-1]["duration_beats"] if trimmed else 0.0

    # 2) write trimmed MIDI at TEMPO_BPM, stem includes source tag so files never collide
    safe_name = display_name.replace(" ", "_").replace("'", "")
    global_idx = idx  # dataset clips occupy 01..08
    stem = f"{global_idx:02d}_{SOURCE}_{idx:02d}_{safe_name}__{style}__{ttype}"
    midi_out = OUT_MIDI / f"{stem}.mid"
    write_trimmed_midi(trimmed, midi_out)

    # 3) render WAV
    wav_out = OUT_AUDIO / f"{stem}.wav"
    sr, n_samples = render_wav(midi_out, wav_out)
    duration_sec = n_samples / sr

    manifest_entries.append({
        "id": f"{SOURCE}_{idx:02d}",
        "source": SOURCE,
        "song": display_name,
        "style": style,
        "transformation": ttype,
        "source_midi": str(src_midi.relative_to(ROOT)),
        "trimmed_midi": str(midi_out.relative_to(ROOT)),
        "audio": str(wav_out.relative_to(ROOT)),
        "bars": N_BARS,
        "beats_per_bar": BEATS_PER_BAR,
        "tempo_bpm": TEMPO_BPM,
        "n_chords": len(trimmed),
        "end_beat": round(actual_end, 2),
        "duration_sec": round(duration_sec, 2),
        "render": {
            "waveform": RENDER_WAVEFORM,
            "speed": RENDER_SPEED,
            "reverb_pct": RENDER_REVERB,
            "sample_rate": sr,
        },
    })

print(f"\nRendered {len(manifest_entries)} clips into {OUT_AUDIO}.")


## Manifest + inline preview

The manifest is written as `dataset/listening_test/manifest.json`. When we add the 14 model clips later, we'll append to the same file so the survey tool reads one list.

Each survey item carries four rating placeholders (Harmony, Plausibility, Dissonance, Novelty) that the downstream test form will populate.

In [ ]:
SURVEY_QUESTIONS = [
    {"key": "harmony",      "label": "Harmony",      "prompt": "How coherent do you find the harmonic motion?"},
    {"key": "plausibility", "label": "Plausibility", "prompt": "How plausible is this chord progression for a potential song?"},
    {"key": "dissonance",   "label": "Dissonance",   "prompt": "How dissonant is this chord progression?"},
    {"key": "novelty",      "label": "Novelty",      "prompt": "How surprising or novel do you find this progression?"},
]

for e in manifest_entries:
    e["ratings"] = {q["key"]: None for q in SURVEY_QUESTIONS}

manifest = {
    "source": SOURCE,
    "tuning": "53-EDO (MPE)",
    "n_bars": N_BARS,
    "tempo_bpm": TEMPO_BPM,
    "styles": sorted({e["style"] for e in manifest_entries}),
    "transformations": sorted({e["transformation"] for e in manifest_entries}),
    "survey_questions": SURVEY_QUESTIONS,
    "clips": manifest_entries,
}

manifest_path = OUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f"wrote {manifest_path}  ({len(manifest_entries)} {SOURCE} clips)")


In [ ]:
# Inline QC — play each clip once to check whether any needs re-selecting.
# for e in manifest_entries:
#     print(f"{e['id']}  {e['song']:24s} [{e['transformation']:17s}] "
#           f"{e['style']:18s} {e['n_chords']:>3d} chords  {e['duration_sec']:5.1f} s")
#     display(Audio(str(ROOT / e["audio"])))

## Next steps

1. **Listen through all 8 clips** above. If any sound muddy / too sparse / repetitive, swap its entry in `SELECTION` — keep style coverage (2 per style) and transformation uniqueness.
2. **Model A clips (8)**: `10_generate_midi_model_A.ipynb` samples 8 progressions — 2 per style × `{jazz, rock, blues, bossa}` with the same 8 transformation labels — and writes them to `dataset/audio/listening_test/model_A/` + `dataset/listening_test/model_A/manifest.json` under the identical render settings (Rhodes, 180 BPM, 17 bars).
3. **Model B clips (8)**: `11_generate_midi_model_B.ipynb` mirrors the above into `.../model_B/`.
4. **Randomisation**: the final 24-clip survey form merges the three manifests and shuffles the ordering per participant using a seed so `source` stays blind.
